In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, auc, roc_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor  
import warnings
import pickle
import os
import json


warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [23]:
cols = ['ts', 'uid', 'id.orig_h', 'id.orig_p',
        'id.resp_h', 'id.resp_p', 'proto', 'service',
        'duration',  'orig_bytes', 'resp_bytes',
        'conn_state', 'local_orig', 'local_resp',
        'missed_bytes',  'history', 'orig_pkts',
        'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
        'tunnel_parents', 'label']

In [19]:
out_dir = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/timeout60'
df1 = pd.read_csv(out_dir+"/UNSW-NB15_zeek_60.csv")

In [20]:
out_dir = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/timeout60'
df2 = pd.read_csv(out_dir+"/UNSW-NB15_zeek_60.csv")
df2.shape

(1028283, 23)

In [21]:
df = pd.concat([df1, df2], ignore_index=True, sort=False)

In [22]:
df = df.drop(columns=['id', 'uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents']) # tunnel_parents is empty

In [23]:
df

,ts,id.orig_p,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,Attack
0,1.424223e+09,61953,24075,tcp,-,0.05984,236,0,S0,F,F,0,Sa,2,112,2,104,Benign
1,1.424223e+09,6881,65360,tcp,-,0.122888,64154,68,SF,F,F,0,DTaTdtAFf,68,81674,37,2009,Benign
2,1.424223e+09,24935,5190,tcp,-,0.004947,216,814,SF,F,F,0,ShADTdtfFa,12,1064,12,2260,Benign
3,1.424223e+09,37415,143,tcp,-,0.137649,68,225,SF,F,F,0,DTadtAfF,8,502,14,1094,Benign
4,1.424223e+09,25718,23652,tcp,-,0.074317,235,22843,SF,F,F,3967,ShdGtAgtfFa,32,1672,35,40046,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2056601,1.421932e+09,1034,520,udp,-,0.000012,1048,0,S0,F,F,0,D,2,1104,0,0,Fuzzers
2056602,1.421928e+09,179,41558,tcp,-,0.074259,0,0,SF,F,F,0,AfFa,6,240,4,160,direction_flip:Fuzzers
2056603,1.421928e+09,1027,42174,tcp,dce_rpc,0.244165,60,6848,SF,F,F,0,DTdtFfaA,6,360,14,14256,direction_flip:Exploits
2056604,1.421931e+09,89,28433,tcp,-,0.021418,0,0,SF,F,F,0,FfaA,4,160,4,160,direction_flip:Exploits


In [33]:
df['Attack'].value_counts()

Benign                           2003182
Exploits                           21030
Fuzzers                            15655
Reconnaissance                      8717
Generic                             3163
DoS                                 3074
Shellcode                           1022
Backdoors                            314
Analysis                             308
Worms                                135
direction_flip:DoS                     2
direction_flip:Exploits                2
direction_flip:Reconnaissance          1
direction_flip:Fuzzers                 1
Name: Attack, dtype: int64

In [36]:
df['Attack'].value_counts()

Benign            2003182
Exploits            21030
Fuzzers             15655
Reconnaissance       8717
Generic              3163
DoS                  3074
Shellcode            1022
Backdoors             314
Analysis              308
Worms                 135
Name: Attack, dtype: int64

In [2]:
def save_scores(timeout, acc, f1, prec, rec):
    results = {
        'Timeout': timeout,
        'accuracy': acc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec
    }

    with open(f'../checkpoints/ET/ET_unsw_zeek_{timeout}.json', 'w+') as f:
        json.dump(results, f, indent=4)

In [8]:
timeouts = ['default', 0.5, 1, 2, 3, 4, 5, 6, 10, 30, 60]

In [3]:
timeouts = [0.5]

In [43]:
best_f1 = 0
best_timeout = None
best_prec = None
best_rec = None
best_acc = None

worst_f1 = 1
worst_acc = None
worst_timeout = None
worst_prec = None
worst_rec = None

save=False

for timeout in timeouts:
    print("Processing timeout : ", timeout)
    if timeout =='default':
        out_dir1 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        out_dir2 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/{timeout}/UNSW-NB15_zeek_{timeout}.csv'
    else:
        out_dir1 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        out_dir2 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}.csv'

    df1 = pd.read_csv(out_dir1)
    df2 = pd.read_csv(out_dir2)
    
    df = pd.concat([df1, df2], ignore_index=True, sort=False)
    
    
    df = df.drop(columns=['id', 'uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents']) # tunnel_parents is empty

    # Handle missing values (if any)
    df.replace({'orig_bytes': '-'}, '0', inplace=True)
    df['orig_bytes'] = pd.to_numeric(df['orig_bytes'], errors='coerce')
    df['orig_bytes'] = df['orig_bytes'].fillna(0).astype('int64')

    df.replace({'resp_bytes': '-'}, '0', inplace=True)
    df['resp_bytes'] = pd.to_numeric(df['resp_bytes'], errors='coerce')
    df['resp_bytes'] = df['resp_bytes'].fillna(0).astype('int64')


    df.replace({'duration': '-'}, '0', inplace=True)
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['duration'] = df['duration'].fillna(0).astype('float64')

    df['service'] = df['service'].replace('-', np.nan)
    df['history'] = df['history'].replace('-', np.nan)

    # Convert categorical variables to numerical using Label Encoding
    # Encode protocol and service types
    label_encoders = {}
    for column in ['proto', 'service', 'conn_state', 'history', 'local_orig', 'local_resp']:
        if column in df.columns:
            le = LabelEncoder()
            df[column] = le.fit_transform(df[column])
            label_encoders[column] = le
    
    df = df[~df['Attack'].str.startswith('direction_flip')]

    # Split df into features and labels
    X = df.drop(columns=['Attack'])  # Assuming 'label' is the target variable
    y = df['Attack']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=None)


    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_test = le.transform(y_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Initialize and train Extra Trees Classifier
    clf = ExtraTreesClassifier(n_estimators=50, random_state=None)
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    #report = classification_report(y_test, pred,  target_names=oe.categories_, digits=4)
    f1 = f1_score(y_true=y_test, y_pred=pred, average='macro')
    if f1 > best_f1: 
        best_timeout = timeout
        best_f1 = f1
        best_pred=pred
        best_y=y_test
print('_______________________________________________________')

Processing timeout :  0.5
_______________________________________________________


In [37]:
best_f1 = 0
best_timeout = None
best_prec = None
best_rec = None
best_acc = None

worst_f1 = 1
worst_acc = None
worst_timeout = None
worst_prec = None
worst_rec = None

save=False

for timeout in timeouts:
    print("Processing timeout : ", timeout)
    if timeout =='default':
        out_dir1 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        out_dir2 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/{timeout}/UNSW-NB15_zeek_{timeout}.csv'
    else:
        out_dir1 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        out_dir2 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}.csv'

    df1 = pd.read_csv(out_dir1)
    df2 = pd.read_csv(out_dir2)
    
    df = pd.concat([df1, df2], ignore_index=True, sort=False)
    
    
    df = df.drop(columns=['uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents']) # tunnel_parents is empty

    # Handle missing values (if any)
    df.replace({'orig_bytes': '-'},  np.nan, inplace=True)
    df.replace({'resp_bytes': '-'},  np.nan, inplace=True)
    df.replace({'duration': '-'},  np.nan, inplace=True)
    df['service'] = df['service'].replace('-', np.nan)
    df['history'] = df['history'].replace('-', np.nan)
    
    df = df.dropna()
    df = df[~df['Attack'].str.startswith('direction_flip')]

    # Convert categorical variables to numerical using Label Encoding
    # Encode protocol and service types
    label_encoders = {}
    for column in ['proto', 'service', 'conn_state', 'history', 'local_orig', 'local_resp']:
        if column in df.columns:
            le = LabelEncoder()
            df[column] = le.fit_transform(df[column])
            label_encoders[column] = le

    # Split df into features and labels
    X = df.drop(columns=['Attack'])  # Assuming 'label' is the target variable
    y = df['Attack']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=None)


    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    y_train = oe.fit_transform(y_train.values.reshape(-1, 1))
    y_test = oe.transform(y_test.values.reshape(-1, 1))



    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Initialize and train Extra Trees Classifier
    clf = ExtraTreesClassifier(n_estimators=100, random_state=None)
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    #report = classification_report(y_test, pred,  target_names=oe.categories_, digits=4)
    f1 = f1_score(y_true=y_test, y_pred=pred, average='macro')
    if f1 > best_f1: 
        best_timeout = timeout
        best_f1 = f1
        best_pred=pred
        best_y=y_test
print('_______________________________________________________')

Processing timeout :  0.5
_______________________________________________________


In [7]:
timeouts = [60]

In [29]:
best_f1 = 0
best_timeout = None
best_prec = None
best_rec = None
best_acc = None

worst_f1 = 1
worst_acc = None
worst_timeout = None
worst_prec = None
worst_rec = None

save=False

for timeout in timeouts:
    print("Processing timeout : ", timeout)
    if timeout =='default':
        out_dir1 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        out_dir2 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/{timeout}/UNSW-NB15_zeek_{timeout}.csv'
    else:
        out_dir1 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/17-02-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        out_dir2 = f'/home/meryem.janati/lustre/nlp_team-um6p-st-sccs-id7fz1zvotk/IDS/janati/IDS/Datasets/unsw/Zeek/22-01-2015/timeout{timeout}/UNSW-NB15_zeek_{timeout}.csv'
        

    df1 = pd.read_csv(out_dir1)
    df2 = pd.read_csv(out_dir2)
    
    df = df1.append(df2, ignore_index=True)
    
    
    df = df.drop(columns=['id', 'uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents']) # tunnel_parents is empty
    #df = df[~df['Attack'].str.startswith('direction_flip')]


    # Handle missing values (if any)
    df.replace({'orig_bytes': '-'}, '0', inplace=True)
    df['orig_bytes'] = pd.to_numeric(df['orig_bytes'], errors='coerce')
    df['orig_bytes'] = df['orig_bytes'].fillna(0).astype('int64')

    df.replace({'resp_bytes': '-'}, '0', inplace=True)
    df['resp_bytes'] = pd.to_numeric(df['resp_bytes'], errors='coerce')
    df['resp_bytes'] = df['resp_bytes'].fillna(0).astype('int64')


    df.replace({'duration': '-'}, '0', inplace=True)
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['duration'] = df['duration'].fillna(0).astype('float64')

    df['service'] = df['service'].replace('-', np.nan)
    df['history'] = df['history'].replace('-', np.nan)
    
    # Convert categorical variables to numerical using Label Encoding
    # Encode protocol and service types
    label_encoders = {}
    for column in ['proto', 'service', 'conn_state', 'history', 'local_orig', 'local_resp']:
        if column in df.columns:
            le = LabelEncoder()
            df[column] = le.fit_transform(df[column])
            label_encoders[column] = le

    # Split df into features and labels
    X = df.drop(columns=['Attack'])  # Assuming 'label' is the target variable
    y = df['Attack']
    
    accuracy, f1, precision, recall =[], [], [], []
    
    skf= StratifiedKFold(n_splits=5,random_state=None)
    skf.get_n_splits(X,y)
    
    for (train_index, test_index), i in zip(skf.split(X, y), range(5)):
        X_train,X_test=X.iloc[train_index],X.iloc[test_index]
        y_train,y_test=y.iloc[train_index],y.iloc[test_index]


        le = LabelEncoder()
        y_train = le.fit_transform(y_train)
        
        y_test = y_test.map(lambda s: '<unknown>' if s not in le.classes_ else s)
        le.classes_ = np.append(le.classes_, '<unknown>')
        y_test = le.transform(y_test)
        
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Initialize and train Extra Trees Classifier
        clf = ExtraTreesClassifier(n_estimators=100, random_state=42) #RandomForestClassifier(random_state=42, class_weight='balanced') 
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        
        f1Score = f1_score(y_true=y_test, y_pred=pred, average='macro')
        accScore=accuracy_score(y_test, pred)
        precScore = precision_score(y_test, pred, average='macro')
        recScrore = recall_score(y_test, pred, average='macro')
                             
        f1.append(f1Score)
        accuracy.append(accScore)
        precision.append(precScore)
        recall.append(recScrore)
        print('Fold: ', i, 'done!')

        
    f1Mean = np.array(f1).mean()
    accMean = np.array(accuracy).mean()
    recMean = np.array(recall).mean()
    precMean = np.array(precision).mean()
    if save:
        save_scores(timeout, accMean, f1Mean, precMean, recMean)

    if f1Mean > best_f1: 
        best_timeout = timeout
        best_f1 = f1Mean
        best_acc = accMean
        best_rec=recMean
        best_prec=precMean
    
    if f1Mean <= worst_f1: 
        worst_timeout = timeout
        worst_f1 = f1Mean
        worst_acc = accMean
        worst_rec=recMean
        worst_prec=precMean
    print('_______________________________________________________')

Processing timeout :  60
Fold:  0 done!
Fold:  1 done!
Fold:  2 done!
Fold:  3 done!
Fold:  4 done!
_______________________________________________________


In [31]:
print("Best Timeout Combination: ", best_timeout)
print("Best Accuracy: ", best_acc)
print('Best Macro F1-score: :', best_f1)
print('Best Macro Precision: :', best_prec)
print('Best Macro Recall: :', best_rec)

Best Timeout Combination:  60
Best Accuracy:  0.9759278143573953
Best Macro F1-score: : 0.41154896733104973
Best Macro Precision: : 0.54307111769675
Best Macro Recall: : 0.3943499647688204


In [16]:
def load_score(timeout):
    with open(f'../checkpoints/ET/ET_unsw_zeek_{timeout}.json', 'r') as f:
        loaded_results = json.load(f)
    return loaded_results


In [17]:
best_f1 = 0
best_timeout = None
best_prec = None
best_rec = None
best_acc = None

worst_f1 = 1
worst_acc = None
worst_timeout = None
worst_prec = None
worst_rec = None


for timeout in timeouts:
    loaded_results = load_score(timeout)
    
    if loaded_results['f1_score'] > best_f1: 
        best_timeout = loaded_results['Timeout'] 
        best_f1 = loaded_results['f1_score'] 
        best_acc = loaded_results['accuracy'] 
        best_rec=loaded_results['recall'] 
        best_prec=loaded_results['precision'] 
    
    if loaded_results['f1_score'] <= worst_f1: 
        worst_timeout = loaded_results['Timeout'] 
        worst_f1 = loaded_results['f1_score'] 
        worst_acc = loaded_results['accuracy'] 
        worst_rec=loaded_results['recall'] 
        worst_prec=loaded_results['precision'] 
    

In [44]:
print("Best Timeout Combination: ", best_timeout)
print("Best Accuracy: ", best_acc)
print('Best Macro F1-score: :', best_f1)
print('Best Macro Precision: :', best_prec)
print('Best Macro Recall: :', best_rec)

Best Timeout Combination:  0.5
Best Accuracy:  None
Best Macro F1-score: : 0.5786802443977858
Best Macro Precision: : None
Best Macro Recall: : None


In [38]:
print("worst Timeout Combination: ", worst_timeout)
print("worst Accuracy: ", worst_acc)
print('worst Macro F1-score: :', worst_f1)
print('worst Macro Precision: :', worst_prec)
print('worst Macro Recall: :', worst_rec)

worst Timeout Combination:  0.5
worst Accuracy:  0.9612934557919635
worst Macro F1-score: : 0.30349321735691015
worst Macro Precision: : 0.47508384930649283
worst Macro Recall: : 0.300484006912235


In [20]:
print("Difference in performance between ", best_timeout, 'and ', worst_timeout)
print("Difference in Accuracy: ", (best_acc - worst_acc)*100)
print('worst Macro F1-score: :', (best_f1 - worst_f1)*100)
print('worst Macro Precision: :', (best_prec - worst_prec)*100)
print('worst Macro Recall: :', (best_rec - worst_rec)*100)

Difference in performance between  5 and  4
Difference in Accuracy:  -0.0003544264523425156
worst Macro F1-score: : 1.393831486366612
worst Macro Precision: : 2.7923704320350295
worst Macro Recall: : 0.908084112501617


In [21]:
import json

results = {
    'Best score': {
        'Best Timeout': best_timeout,
        'Accuracy': best_acc,
        'F1 Score': best_f1,
        'Precision': best_prec,
        'Recall': best_rec
    },
    
    'Worst score': {
        'Worst Timeout': worst_timeout,
        'Accuracy': worst_acc,
        'F1 Score': worst_f1,
        'Precision': worst_prec,
        'Recall': worst_rec
    },
    
    'Difference': {
        'Accuracy': (best_acc - worst_acc)*100,
        'F1 Score': (best_f1 - worst_f1)*100,
        'Precision': (best_prec - worst_prec)*100,
        'Recall': (best_rec - worst_rec)*100
    }
}



with open('../results/ET_unsw_zeek.json', 'w') as f:
    json.dump(results, f, indent=4)